# Chapter 3 — Define the Contract

**Book alignment:** DSPy From First Principles, Chapter 3

**Question this notebook isolates:** Do field names, the input boundary, and output types determine what the model sees and what downstream code can check — separately from prompt wording?


In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "experiments" / "dspy-from-first-principles" / "common").exists():
            return candidate
    raise RuntimeError(
        "Run this notebook from a checkout containing experiments/dspy-from-first-principles"
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
EXP_ROOT = REPO_ROOT / "experiments" / "dspy-from-first-principles"
sys.path.insert(0, str(EXP_ROOT))

import dspy
from common.data import teaching_cases
from common.dspy_program import RewriteSentence, to_dspy_example
from common.judge import SemanticConstraintJudge


## Compact form versus class form

Both spellings produce working signatures over the same task-facing core. Only the class form declares the instruction, per-field descriptions, the extra analysis inputs, and a reusable symbol — the meaning a reviewer can audit rather than infer.


In [ ]:
compact = dspy.Predict("sentence, goal, context -> rewritten_text, rationale")
print("compact inputs:", sorted(compact.signature.input_fields))
print("class inputs:  ", sorted(RewriteSentence.input_fields))
print("class outputs: ", sorted(RewriteSentence.output_fields))
print("instruction:   ", (RewriteSentence.__doc__ or "").strip().splitlines()[0])


In [ ]:
assert set(compact.signature.output_fields) == set(RewriteSentence.output_fields)
assert set(compact.signature.input_fields) < set(RewriteSentence.input_fields)
assert (RewriteSentence.__doc__ or "").strip() != ""
print("same task-facing core; only the class form carries declared meaning")


## The input boundary is mechanical

`.with_inputs("sentence", "goal", "context")` is the one line that separates what the program may see from what evaluation may see. Reference rewrites, constraints, and provenance stay attached to the row without becoming task inputs.


In [ ]:
ed003 = {c.case_id: c for c in teaching_cases()}["ed-003"]
example = to_dspy_example(ed003)
program_inputs = dict(example.inputs())
print("program inputs:", sorted(program_inputs))
print("reference attached:", hasattr(example, "reference_rewrite"))
print("constraints attached:", hasattr(example, "semantic_constraints"))


In [ ]:
assert set(program_inputs) == {"sentence", "goal", "context"}
assert "reference_rewrite" not in program_inputs
assert "semantic_constraints" not in program_inputs
assert example.reference_rewrite.startswith('"I do not think')
print("generation sees three fields; evaluation keeps the answers")


## Every output field needs a job

Types surface failures that prose hides — but only where a closed vocabulary earns its keep. The semantic judge gets `Literal` verdicts because downstream code aggregates over them; the rewrite program drops `confidence` because no consumer or calibration exists.


In [ ]:
import typing

verdict_values = typing.get_args(SemanticConstraintJudge.model_fields["verdict"].annotation)
reason_values = typing.get_args(SemanticConstraintJudge.model_fields["reason_code"].annotation)
print("verdict vocabulary:", verdict_values)
print("reason codes:", len(reason_values))
print("rewrite outputs:", sorted(RewriteSentence.output_fields))

FIELD_JOBS = {
    "rewritten_text": ("application output; evaluation target", True),
    "rationale": ("diagnostic only; never acceptance evidence", False),
}
for field, (job, controls) in FIELD_JOBS.items():
    print(f"{field}: {job} | controls action: {controls}")


In [ ]:
assert set(verdict_values) == {"preserved", "violated", "unclear"}
assert "uncertainty is representable" if "unclear" in verdict_values else False
assert "confidence" not in RewriteSentence.output_fields
assert "risk" not in RewriteSentence.output_fields
print("typed where code branches; deleted where nothing consumes")


## What we earned

The contract now names what the model consumes and produces in the application's own concepts. Field names are the durable part of the optimization surface; the input projection keeps answers out of generation; output fields survive only with a consumer and a check.

Notebook 04 / Chapter 4 holds that contract completely fixed and changes only the strategy: does asking the model to reason before it answers produce a better rewrite, and what does it cost?
